In [1]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1380 documents


In [2]:
type(documents)

list

In [3]:
type(documents[0])

dict

Therefore, documents is a list of dictionaries!

In [4]:
# For example, you can access the hundred & first document like this:

documents[100]

{'id': '1fda7c57b0',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 2. Machine Learning for Regression',
 'question': 'ValueError: shapes not aligned',
 'answer': '```python\nX_train = prepare_X(df_train)\nw_0, w = train_linear_regression(X_train, y_train)\n\nX_val = prepare_X(df_val)\ny_pred = w_0 + X_val.dot(w)\n\nrmse(y_val, y_pred)\n```\n\nWe get:\n\n```\nValueError                                Traceback (most recent call last)\nInput In [132], in <cell line: 5>()\n      2 w_0, w = train_linear_regression(X_train, y_train)\n      4 X_val = prepare_X(df_val)\n----> 5 y_pred = w_0 + X_val.dot(w)\n      7 rmse(y_val, y_pred)\n\nValueError: shapes (4128,) and (1,) not aligned: 4128 (dim 0) != 1 (dim 0)\n```\n\nIf we try to perform an arithmetic operation between two arrays of different shapes or dimensions, it throws an error like operands could not be broadcast together with shapes. Broadcasting can occur in certain scenarios and will fail in others.\n\nTo solve this is

In [5]:
# filter documents and keep only those that are part of the LLM Zoomcamp course

docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 118 documents


In [6]:
# For example, you can access the hundred & first document like this:

docs_llm[100]

{'id': 'd4f7c08ea1',
 'course': 'llm-zoomcamp',
 'section': 'Capstone Project',
 'question': 'Project: do I need an orchestration tool (Airflow, Mage, Kestra) for the capstone?',
 'answer': 'No. A plain Python script that ingests and indexes your data is enough for full points on the "ingestion pipeline" criterion. A Jupyter notebook with the same steps is worth 1 point instead of 2.\n\nUse an orchestrator only if it actually fits your project — for example, recurring ingestion of a feed that updates daily. Don\'t add one just to score the criterion.'}

In [7]:
# Now create a SQLitesearch index for the LLM Zoomcamp documents

# It configures an index with three settings:

# text_fields: fields searched using full-text search. 
# A query can match words found in an FAQ’s question, section, or answer.
    
# keyword_fields: fields intended for exact filtering rather than free-text matching. 
# For example, course="machine-learning-zoomcamp" could limit results to one course.

# db_path: the SQLite database file used to store the index. 
# If appropriate, faq_llm.db will be created in the program’s current working directory.

# The resulting sqlite_index object is configured but initially contains no FAQ records. 
# Data must be added before searches return results.

# Conceptually, each indexed record might look like:
# {
#     "question": "How do I install Docker?",
#     "section": "Setup",
#     "answer": "Download Docker Desktop...",
#     "course": "machine-learning-zoomcamp"
# }

# A search for "install Docker" would examine the three text fields, 
# while the course field could be used to filter the matches.


import time
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq_llm.db"  # Path to the SQLite database that will be created to store the index
)

for doc in docs_llm:
    sqlite_index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    #time.sleep(0.5) # slow down the ingestion to illustrate the process

sqlite_index.close()
print("Done. Index saved to faq_llm.db")

Added: I just discovered the course. Can I still join?...
Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: How should I start the course and follow the weekly workflow...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: Where can I track the LLM Zoomcamp syllabus, deadlines, home...
Added: Are there live sessions or office hours for each module?...
Added: Can I use Bluesky for learning in public credits?...
Added: Where is the LLM Zoomcamp Telegram channel?...
Added: Why doesn't the number of records I get in the FAQ dataset m...
Added: The homework submission f

In [8]:
sqlite_index.count()

1220

In [9]:
sqlite_index

In [10]:
# Now that the SQLite index is created, we can query it to find relevant FAQ records.

sqlite_index.search("install Docker")

[{'id': '66ccbb7da0',
  'course': 'llm-zoomcamp',
  'section': 'Module 5: Monitoring',
  'question': 'How can I remove all Docker containers, images, and volumes, and builds from the terminal?',
  'answer': '1. Delete all containers (including running ones):\n\n```bash\ndocker rm -f $(docker ps -aq)\n```\n\n2. Remove all images:\n\n```bash\ndocker rmi -f $(docker images -q)\n```\n\n3. Delete all volumes:\n\n```bash\ndocker volume rm $(docker volume ls -q)\n```'},
 {'id': '66ccbb7da0',
  'course': 'llm-zoomcamp',
  'section': 'Module 5: Monitoring',
  'question': 'How can I remove all Docker containers, images, and volumes, and builds from the terminal?',
  'answer': '1. Delete all containers (including running ones):\n\n```bash\ndocker rm -f $(docker ps -aq)\n```\n\n2. Remove all images:\n\n```bash\ndocker rmi -f $(docker images -q)\n```\n\n3. Delete all volumes:\n\n```bash\ndocker volume rm $(docker volume ls -q)\n```'},
 {'id': '66ccbb7da0',
  'course': 'llm-zoomcamp',
  'section': '

In [11]:
results = sqlite_index.search?


Signature:
sqlite_index.search(
    query: str,
    filter_dict: dict[str, typing.Any] | None = None,
    boost_dict: dict[str, float] | None = None,
    num_results: int = 10,
    output_ids: bool = False,
) -> list[dict[str, typing.Any]]
Docstring:
Search the index with the given query.

Args:
    query: The search query string. Supports FTS5 query syntax.
    filter_dict: Dictionary of filters. Can include:
        - Keyword fields: {"field": "value"} for exact match
        - Keyword fields: {"field": ["a", "b"]} for IN/OR (match any value)
        - Numeric fields: {"field": [('>=', 100), ('<', 200)]} for range filters
        - Numeric fields: {"field": 100} for exact match
        - Date fields: {"field": [('>=', date(...)), ('<', date(...))]} for range filters
        - Any field: {"field": None} for null/missing values
    boost_dict: Dictionary of boost scores for text fields.
    num_results: Maximum number of results to return.
    output_ids: If True, adds an 'id' field wi

In [12]:
results = sqlite_index.search(
    query="How do I submit homework?",
    boost_dict={
        "question": 3.0,
        "section": 1.0,
        "answer": 1.0,
    },
    num_results=5,
)

results

[{'id': 'cdc3b285e5',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Can I submit homework after the deadline, or get a deadline extension?',
  'answer': "No. We don't give individual deadline extensions, and once the homework submission form is closed you can no longer submit it — there are no late submissions. While the form is still open you can submit, even if the listed deadline has already passed.\n\nMissing a homework won't affect your certificate: homework isn't mandatory, only passing the Capstone project is. Homework points only count toward your leaderboard rank, so you'll still appear on the leaderboard with your other submissions."},
 {'id': 'cdc3b285e5',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Can I submit homework after the deadline, or get a deadline extension?',
  'answer': "No. We don't give individual deadline extensions, and once the homework submission form is closed yo

In [13]:
# Now let's query the SQLite database to see what we have stored.

import sqlite3

with sqlite3.connect("faq_llm.db") as connection:
    connection.row_factory = sqlite3.Row

    records = connection.execute(
        "SELECT * FROM docs LIMIT 10"
        #"SELECT doc_json AS doc_table FROM docs LIMIT 10"    
        #"SELECT * FROM doc_table WHERE question LIKE '%Docker%' OR section LIKE '%Docker%' OR answer LIKE '%Docker%' LIMIT 10"
    ).fetchall()

    for record in records:
        print(type(record))
        print(dict(record))
        #print(record["doc_json"])

<class 'sqlite3.Row'>
{'id': 1, 'doc_json': '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}', 'vector_hash': None, 'course': 'llm-zoomcamp'}
<class 'sqlite3.Row'>
{'id': 2, 'doc_json': '{"id": "977bf7786c", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?", "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}', 'vector_hash': None, 'course': 'llm-zoomcamp'}
<class 'sqlite3.Row'>
{'id': 3, 'doc_json':

In [14]:
# doc_json looks like a dictionary, but it is currently a string:

print(type(record["doc_json"]))
# <class 'str'>

<class 'str'>


In [15]:
# Convert it back into a Python dictionary with json.loads()

import json

for record in records:
    doc_dict = json.loads(record["doc_json"])
    print(type(doc_dict))
    print(doc_dict)

<class 'dict'>
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}
<class 'dict'>
{'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}
<class 'dict'>
{'id': '489dd1c9d9', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessi

In [16]:
# Now let's print the individual fields of each document in a more readable format.

for record in records:
    document = json.loads(record["doc_json"])

    print("Document ID:", document["id"])
    print("Course:", document["course"])
    print("Section:", document["section"])
    print("Question:", document["question"])
    print("Answer:", document["answer"])
    print()

Document ID: 74eb249bbf
Course: llm-zoomcamp
Section: General Course-Related Questions
Question: I just discovered the course. Can I still join?
Answer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Document ID: 977bf7786c
Course: llm-zoomcamp
Section: General Course-Related Questions
Question: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
Answer: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Document ID: 489dd1c9d9
Course: llm-zoomcamp
Section: General Course-Related Questions
Question: What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
Answer: The zoom link is only published to instructors/presenters/TAs.

Students p

In [17]:
# Now let's query the SQLite database to see what tables we have stored.

with sqlite3.connect("faq_llm.db") as connection:
    tables = connection.execute(
        "SELECT name FROM sqlite_master WHERE type = 'table'"
    ).fetchall()

    for table in tables:
        print(table[0])

docs
sqlite_sequence
docs_fts
docs_fts_data
docs_fts_idx
docs_fts_content
docs_fts_docsize
docs_fts_config


In [18]:
# Now let's query the SQLite database to see what we have stored.

with sqlite3.connect("faq_llm.db") as connection:
    connection.row_factory = sqlite3.Row

    records = connection.execute("""
        SELECT *
        FROM docs
        LIMIT 10
    """).fetchall()

    for record in records:
        print(dict(record))

{'id': 1, 'doc_json': '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}', 'vector_hash': None, 'course': 'llm-zoomcamp'}
{'id': 2, 'doc_json': '{"id": "977bf7786c", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?", "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}', 'vector_hash': None, 'course': 'llm-zoomcamp'}
{'id': 3, 'doc_json': '{"id": "489dd1c9d9", "course": "llm-zoomcamp", "section": "Gener

In [19]:
# Now let's query the SQLite database to see what columns we have stored.

with sqlite3.connect("faq_llm.db") as connection:
    columns = connection.execute(
        "PRAGMA table_info(docs)"
    ).fetchall()

    for column in columns:
        print(column)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'doc_json', 'TEXT', 1, None, 0)
(2, 'vector_hash', 'BLOB', 0, None, 0)
(3, 'course', 'TEXT', 0, None, 0)


In [20]:
# You can also examine one particular record by its internal ID:

with sqlite3.connect("faq_llm.db") as connection:
    connection.row_factory = sqlite3.Row

    record = connection.execute(
        "SELECT * FROM docs WHERE id = ?",
        (1,)
    ).fetchone()

    print(dict(record) if record else "Record not found")

{'id': 1, 'doc_json': '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}', 'vector_hash': None, 'course': 'llm-zoomcamp'}


In [21]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [22]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [23]:
sqlite_index.close()

### Now, let's try ElasticSearch...

In [24]:
# Now let's try to connect to an Elasticsearch instance and see if we can get some information about it.

from elasticsearch import Elasticsearch

es_client = Elasticsearch("http://localhost:9200")

print(es_client.info())

{'name': '72e420ec1373', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'yLbNFiliSwWTTXrOgj3w-w', 'version': {'number': '8.17.6', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'dbcbbbd0bc4924cfeb28929dc05d82d662c527b7', 'build_date': '2025-04-30T14:07:12.231372970Z', 'build_snapshot': False, 'lucene_version': '9.12.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [32]:
# Firstly, we will reload the FAQ data from the JSON file using the `load_faq_data` function 
# from the `ingest` module. This function reads the data and returns a list of documents.

from ingest import load_faq_data

docs_faq = load_faq_data()
print(f"Loaded {len(docs_faq)} documents")

Loaded 1380 documents


In [33]:
# Now let's create an Elasticsearch index for all the Zoomcamp FAQ documents

index_name = "faq-zoomcamp"

mapping = {
    "mappings": {
        "properties": {
            "question": {"type": "text"},
            "section": {"type": "text"},
            "answer": {"type": "text"},
            "course": {"type": "keyword"},
        }
    }
}

# Check if the index already exists before creating it.
# In the following code snippet, '**' is used to unpack the mapping dictionary into keyword arguments for the create method. 
# This allows the mapping to be passed as a single argument rather than as separate keyword arguments.

if not es_client.indices.exists(index=index_name):
    es_client.indices.create(
        index=index_name,
        **mapping,
    )

In [34]:
# Now let's index the Zoomcamp documents into Elasticsearch.

# Bulk indexing is a more efficient way to index multiple documents at once, 
# rather than indexing them one by one.

from elasticsearch.helpers import bulk

# The following code snippet uses a generator expression to create a sequence of actions 
# for the bulk indexing operation.

# The generator expression iterates over the docs_llm list and creates a dictionary for each document.
# Each dictionary contains the index name, document ID, and the document itself as the source.

# The use of a generator expression allows for efficient memory usage, 
# as it generates each action on-the-fly rather than storing them all in memory at once.
# Parentheses '()' are used to create a generator expression, which is a concise way to create an iterator 
# that yields items one at a time.

# Square brackets '[]' would create a list, which would store all items in memory at once.
# Generators are normally single-use, meaning that once they are exhausted, they cannot be reused.

# (...)  = generator expression
# [...]  = list comprehension

actions = (
    {
        "_index": index_name,
        "_id": document["id"],  # Use the document's ID as the Elasticsearch document ID, to avoid duplicates
        "_source": document,
    }
    for document in docs_faq
)

indexed_count, errors = bulk(es_client, actions)

print(f"Indexed {indexed_count} documents")
print(f"Errors: {errors}")

Indexed 1380 documents
Errors: []


In [35]:
# Now let's refresh the index to make sure all documents are searchable immediately after indexing.

es_client.indices.refresh(index=index_name)

ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

In [36]:
# Now let's query the Elasticsearch index to see what we have stored.

response = es_client.search(
    index=index_name,
    size=5,
    query={
        "multi_match": {
            "query": "Can I still join the course?",
            "fields": [
                "question^3",   # Boost the question field to give it more weight in the search results
                "section",
                "answer",
            ],
        }
    },
)

for hit in response["hits"]["hits"]:
    print("Score:", hit["_score"])
    print("Course:", hit["_source"]["course"])
    print("Question:", hit["_source"]["question"])
    print("Answer:", hit["_source"]["answer"])
    print()

Score: 57.36765
Course: llm-zoomcamp
Question: I just discovered the course. Can I still join?
Answer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Score: 56.022366
Course: mlops-zoomcamp
Question: Course - Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.

Score: 56.022366
Course: data-engineering-zoomcamp
Question: Course: Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

Score: 53.62921
Course: machine-lear

In [39]:
# Now let's query the Elasticsearch index to see what we have stored, but this time with a filter for the course.

response = es_client.search(
    index=index_name,
    size=5,
    query={
        "bool": {
            "must": {
                "multi_match": {
                    "query": "homework submission",
                    "fields": [
                        "question^3",
                        "section",
                        "answer",
                    ],
                }
            },
            "filter": {
                "term": {
                    "course": "llm-zoomcamp"
                }
            },
        }
    },
)

for hit in response["hits"]["hits"]:
    print("Score:", hit["_score"])
    print("Course:", hit["_source"]["course"])
    print("Question:", hit["_source"]["question"])
    print("Answer:", hit["_source"]["answer"])
    print()

Score: 21.303162
Course: llm-zoomcamp
Question: The homework submission form is still open even though the deadline has passed — can I still submit?
Answer: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

Score: 18.994167
Course: llm-zoomcamp
Question: My homework submission is rejected because my repo URL returns a non-200 status (e.g. 500) — how do I fix it?
Answer: The submission checker fetches the URL you submit with a GET request and expects an HTTP `200` response. A non-200 status (404, 500, etc.) almost always means the link isn't publicly reachable as-is. Check the usual causes:

- **Private repo** — make the repository public so the checker can access it.
- **Trailing `.git`** — submit the plain repository URL (e.g. `https://github.com/you/repo`), not `https://github.com/you/repo.git`.
- 

In [40]:
es_client.close()

### Now, let's test the persistence of the ElasticSearch index...

#### Persistence is enabled via the docker-compose.yml config file...

```yaml
services:
  elasticsearch:
    volumes:
      - elasticsearch-data:/usr/share/elasticsearch/data

volumes:
  elasticsearch-data:
```


In [41]:
# Restart the Elasticsearch service

!docker compose restart elasticsearch

[+] Restarting 0/1
 ⠋ Container elasticsearch  Restarting                                     0.0s 
[+] Restarting 0/1
 ⠙ Container elasticsearch  Restarting                                     0.1s 
[+] Restarting 0/1
 ⠹ Container elasticsearch  Restarting                                     0.2s 
[+] Restarting 0/1
 ⠸ Container elasticsearch  Restarting                                     0.3s 
[+] Restarting 0/1
 ⠼ Container elasticsearch  Restarting                                     0.4s 
[+] Restarting 0/1
 ⠴ Container elasticsearch  Restarting                                     0.5s 
[+] Restarting 0/1
 ⠦ Container elasticsearch  Restarting                                     0.6s 
[+] Restarting 0/1
 ⠧ Container elasticsearch  Restarting                                     0.7s 
[+] Restarting 0/1
 ⠇ Container elasticsearch  Restarting                                     0.8s 
[+] Restarting 0/1
 ⠏ Container elasticsearch  Restarting                                     0.9s 


In [43]:
# Now let's try to connect to an Elasticsearch instance and see if we can get some information about it.

from elasticsearch import Elasticsearch

es_client = Elasticsearch("http://localhost:9200")

assert es_client.ping()
print("Connected")

Connected


In [45]:
# Now let's query the Elasticsearch index to see if we can retrieve a specific document by its ID.

response = es_client.search(
    index=index_name,
    size=5,
    query={
        "multi_match": {
            "query": "Can I still join the course?",
            "fields": [
                "question^3",   # Boost the question field to give it more weight in the search results
                "section",
                "answer",
            ],
        }
    },
)

for hit in response["hits"]["hits"]:
    print("Score:", hit["_score"])
    print("Course:", hit["_source"]["course"])
    print("Question:", hit["_source"]["question"])
    print("Answer:", hit["_source"]["answer"])
    print()

Score: 57.36765
Course: llm-zoomcamp
Question: I just discovered the course. Can I still join?
Answer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Score: 56.022366
Course: mlops-zoomcamp
Question: Course - Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.

Score: 56.022366
Course: data-engineering-zoomcamp
Question: Course: Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

Score: 53.62921
Course: machine-lear

In [46]:
es_client.close()